# Electivo de Bioinformática — Clase 1

## Conexión a servidor remoto y manejo básico de la línea de comandos en Linux

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 4 de agosto

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Explicar qué es un clúster de cómputo y por qué se usa en bioinformática.
2. Conectarse de forma segura al clúster de la UDD vía SSH.
3. Navegar el sistema de archivos de Linux desde la línea de comandos.
4. Crear, mover, copiar, editar y eliminar archivos y directorios.
5. Usar redirección, tuberías (`pipes`) y comandos de búsqueda para procesar archivos de texto.
6. Interpretar permisos de archivos y gestionar procesos básicos.

### Estructura de la sesión (3 horas)

| Bloque | Tema | Tiempo aprox. |
|---|---|---|
| 1 | Clústers y computación remota en bioinformática | 15 min |
| 2 | Conceptos de SSH y buenas prácticas de seguridad | 20 min |
| 3 | Conexión práctica al clúster UDD | 30 min |
| — | *Pausa* | 10 min |
| 4 | Línea de comandos de Linux: fundamentos | 60 min |
| 5 | Permisos, procesos y variables de entorno | 30 min |
| 6 | Ejercicio integrador con datos tipo FASTQ | 25 min |
| 7 | Cierre, resumen y tarea | 10 min |

> **Nota sobre este notebook:** las celdas de código usan la magia `%%bash`, por lo que se ejecutan con un kernel de Python que tiene bash disponible. Esto les permite **practicar los comandos ahora mismo**, sin necesidad de estar conectados al clúster. En la Sección 3 encontrarán las instrucciones exactas para replicar lo mismo mediante una conexión SSH real al servidor de la UDD.


---
## 1. Clústers y computación remota en bioinformática

Los análisis bioinformáticos (alineamiento de secuencias, llamado de variantes, RNA-Seq, etc.) suelen requerir:

- **Recursos computacionales** muy superiores a los de un computador personal (decenas de CPUs, cientos de GB de RAM).
- **Almacenamiento** de gran capacidad para archivos crudos (FASTQ, BAM) que pueden pesar decenas o cientos de GB por muestra.
- **Reproducibilidad**: un entorno compartido y controlado facilita que distintos investigadores obtengan los mismos resultados.
- **Ejecución de trabajos de larga duración** (horas o días), que no es práctico dejar corriendo en un laptop.

Un **clúster** es un conjunto de computadores (*nodos*) interconectados que se administran como un solo sistema. Habitualmente distinguimos:

- **Nodo de acceso / login node**: donde uno se conecta al ingresar (SSH), edita archivos y prepara trabajos. *No* se deben correr análisis pesados aquí.
- **Nodos de cómputo / compute nodes**: donde efectivamente corren los análisis, generalmente a través de un **gestor de colas** (Slurm, PBS, SGE — lo veremos en clases posteriores cuando enviemos trabajos pesados).
- **Sistema de archivos compartido**: visible desde todos los nodos (p. ej. `/home`, `/scratch`, `/data`).

Hoy nos enfocamos en el primer paso indispensable: **llegar** al clúster y **movernos** dentro de él.


---
## 2. Conceptos de SSH

**SSH (Secure Shell)** es el protocolo estándar para conectarse de forma segura (cifrada) a un servidor remoto y operar su línea de comandos como si estuviéramos sentados frente a él.

### Elementos de una conexión SSH

```
ssh usuario@servidor.udd.cl
```

- `usuario`: su nombre de usuario en el clúster (entregado por el equipo de TI/soporte del curso).
- `servidor.udd.cl`: la dirección (hostname o IP) del clúster.
- Puerto por defecto: `22` (si el servidor usa otro puerto: `ssh -p <puerto> usuario@servidor`).

### Autenticación: contraseña vs. llave pública/privada

| Método | Cómo funciona | Ventajas |
|---|---|---|
| Contraseña | Se ingresa la clave de la cuenta en cada conexión | Simple, pero menos seguro y molesto para uso frecuente |
| Par de llaves SSH | Se genera un par (`clave privada` en su computador + `clave pública` en el servidor) | Más seguro, permite conexión sin escribir contraseña, requerido por muchos clústers |

### Generar un par de llaves (una sola vez, en su propio computador)

```bash
ssh-keygen -t ed25519 -C "su_correo@udd.cl"
```

Esto crea (por defecto) dos archivos en `~/.ssh/`:

- `id_ed25519` → **clave privada**: nunca se comparte, nunca se sube a ningún repositorio.
- `id_ed25519.pub` → **clave pública**: esta sí se entrega/copia al servidor.

### Copiar la llave pública al servidor

```bash
ssh-copy-id usuario@servidor.udd.cl
```

Si `ssh-copy-id` no está disponible (p. ej. en Windows sin WSL), se puede copiar el contenido de `id_ed25519.pub` manualmente a `~/.ssh/authorized_keys` en el servidor.

### Buenas prácticas de seguridad

- Nunca compartan su clave privada ni la suban a GitHub u otro repositorio.
- Usen una *passphrase* al generar la llave (capa adicional de protección).
- Cierren sesión (`exit` o `logout`) al terminar, especialmente en computadores compartidos.
- Reporten cualquier acceso o comportamiento sospechoso al equipo de soporte.


---
## 3. Conexión práctica al clúster de la UDD

Sigan estos pasos **desde la terminal de su propio computador** (no desde este notebook — este notebook corre localmente, la conexión SSH la harán en su terminal habitual: Terminal en Mac/Linux, o PowerShell/WSL en Windows).

### Paso a paso

1. Abrir una terminal.
2. Verificar conectividad de red (deben estar en la red UDD o VPN, según indique el profesor):
   ```bash
   ping servidor.udd.cl
   ```
3. Conectarse:
   ```bash
   ssh usuario@servidor.udd.cl
   ```
4. Si es la primera vez que se conectan, verán un mensaje sobre la *fingerprint* del servidor — escriban `yes` para continuar y confiar en el host.
5. Ingresar la contraseña (o, si ya configuraron la llave, debería entrar directamente).
6. Una vez dentro, confirmar dónde están y quiénes son:
   ```bash
   whoami
   hostname
   pwd
   uname -a
   ```
7. Revisar espacio disponible y recursos del sistema:
   ```bash
   df -h ~
   free -h
   nproc
   ```
8. Para salir de la sesión remota:
   ```bash
   exit
   ```

### Errores comunes

| Mensaje / síntoma | Causa probable | Solución |
|---|---|---|
| `Connection refused` | Servidor caído, puerto incorrecto, o fuera de la red/VPN | Verificar red/VPN y puerto |
| `Permission denied (publickey,password)` | Usuario o contraseña incorrectos, o llave no autorizada | Revisar credenciales con soporte |
| `Connection timed out` | Problema de red o firewall | Verificar conexión a internet / VPN |
| Conexión se corta sola | Timeout por inactividad | Usar `tmux` o `screen` (lo veremos más adelante) para sesiones persistentes |

> **Tarea para antes de la próxima clase:** si aún no tienen credenciales o no lograron conectarse, avisen al equipo docente lo antes posible.


---
## 4. Línea de comandos de Linux: fundamentos

A partir de aquí, **sí pueden ejecutar las celdas de código de este notebook** para practicar. Los mismos comandos funcionan idénticamente una vez conectados al clúster por SSH.

### 4.1 El sistema de archivos y la navegación

Linux organiza todo en una única jerarquía de directorios que comienza en `/` (raíz).

- Ruta **absoluta**: comienza desde `/`, p. ej. `/home/usuario/datos`.
- Ruta **relativa**: parte desde el directorio actual, p. ej. `datos/muestra1`.
- `~` es un atajo para el directorio *home* del usuario.
- `.` es el directorio actual; `..` es el directorio padre.

Comandos clave: `pwd`, `ls`, `cd`.


In [1]:
%%bash
# ¿Dónde estoy?
pwd

# ¿Qué hay en este directorio?
ls -la


/content
total 16
drwxr-xr-x 1 root root 4096 Jun  4 13:32 .
drwxr-xr-x 1 root root 4096 Aug  3 21:45 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [2]:
%%bash
# Creemos un espacio de trabajo para practicar durante la clase
mkdir -p ~/curso_bioinfo/clase1
cd ~/curso_bioinfo/clase1
pwd
ls -la


/root/curso_bioinfo/clase1
total 8
drwxr-xr-x 2 root root 4096 Aug  3 21:58 .
drwxr-xr-x 3 root root 4096 Aug  3 21:58 ..


### 4.2 Crear, copiar, mover y eliminar archivos y directorios

| Comando | Uso |
|---|---|
| `mkdir nombre` | Crear directorio |
| `mkdir -p a/b/c` | Crear directorios anidados |
| `touch archivo` | Crear archivo vacío / actualizar fecha |
| `cp origen destino` | Copiar archivo |
| `cp -r origen destino` | Copiar directorio (recursivo) |
| `mv origen destino` | Mover o renombrar |
| `rm archivo` | Eliminar archivo (¡sin papelera, cuidado!) |
| `rm -r directorio` | Eliminar directorio recursivamente |
| `rmdir directorio` | Eliminar directorio vacío |

> ⚠️ **`rm` es irreversible en la línea de comandos.** No existe un "restaurar desde la papelera". Cuando usen `rm -rf`, revisen dos veces la ruta antes de ejecutar.


In [3]:
%%bash
cd ~/curso_bioinfo/clase1

# Crear estructura de un proyecto típico
mkdir -p proyecto_demo/{raw_data,scripts,results}
touch proyecto_demo/scripts/analisis.sh
touch proyecto_demo/raw_data/muestra1.txt

# Ver el árbol de directorios creado
find proyecto_demo


proyecto_demo
proyecto_demo/raw_data
proyecto_demo/raw_data/muestra1.txt
proyecto_demo/results
proyecto_demo/scripts
proyecto_demo/scripts/analisis.sh


In [4]:
%%bash
cd ~/curso_bioinfo/clase1/proyecto_demo

# Copiar y mover
cp raw_data/muestra1.txt raw_data/muestra1_copia.txt
mv raw_data/muestra1_copia.txt results/

ls -la raw_data/
ls -la results/


total 8
drwxr-xr-x 2 root root 4096 Aug  3 21:59 .
drwxr-xr-x 5 root root 4096 Aug  3 21:58 ..
-rw-r--r-- 1 root root    0 Aug  3 21:58 muestra1.txt
total 8
drwxr-xr-x 2 root root 4096 Aug  3 21:59 .
drwxr-xr-x 5 root root 4096 Aug  3 21:58 ..
-rw-r--r-- 1 root root    0 Aug  3 21:59 muestra1_copia.txt


### 4.3 Ver el contenido de archivos

| Comando | Uso |
|---|---|
| `cat archivo` | Mostrar todo el contenido |
| `less archivo` | Ver contenido con paginación (interactivo; `q` para salir) |
| `head -n N archivo` | Primeras N líneas |
| `tail -n N archivo` | Últimas N líneas |
| `tail -f archivo` | Seguir un archivo en tiempo real (útil para logs) |
| `wc -l archivo` | Contar líneas |

Vamos a generar un archivo de texto simulando un formato bioinformático simple (una tabla de conteos) para practicar.


In [5]:
%%bash
cd ~/curso_bioinfo/clase1/proyecto_demo/raw_data

# Generamos una tabla simulada de expresión génica (gen, conteo)
cat << 'EOF' > conteos_simulados.txt
gen_id	muestra1	muestra2
BRCA1	120	98
TP53	340	310
EGFR	45	52
MYC	670	590
GAPDH	5200	5100
ACTB	4980	5050
EOF

echo "--- Archivo completo (cat) ---"
cat conteos_simulados.txt

echo -e "\n--- Primeras 3 líneas (head) ---"
head -n 3 conteos_simulados.txt

echo -e "\n--- Número de líneas (wc -l) ---"
wc -l conteos_simulados.txt


--- Archivo completo (cat) ---
gen_id	muestra1	muestra2
BRCA1	120	98
TP53	340	310
EGFR	45	52
MYC	670	590
GAPDH	5200	5100
ACTB	4980	5050

--- Primeras 3 líneas (head) ---
gen_id	muestra1	muestra2
BRCA1	120	98
TP53	340	310

--- Número de líneas (wc -l) ---
7 conteos_simulados.txt


### 4.4 Búsqueda: `grep` y `find`

- `grep patrón archivo`: busca líneas que contienen un patrón.
- `grep -i`: ignora mayúsculas/minúsculas.
- `grep -v`: muestra líneas que **no** coinciden.
- `grep -c`: cuenta coincidencias.
- `find directorio -name "patrón"`: busca archivos por nombre.


In [6]:
%%bash
cd ~/curso_bioinfo/clase1/proyecto_demo/raw_data

echo "--- Buscar el gen TP53 ---"
grep "TP53" conteos_simulados.txt

echo -e "\n--- Genes con conteo alto en muestra1 (ejemplo simple con grep -E) ---"
grep -E "	[0-9]{3,}" conteos_simulados.txt

echo -e "\n--- Buscar archivos .txt en todo el proyecto ---"
find .. -name "*.txt"


--- Buscar el gen TP53 ---
TP53	340	310

--- Genes con conteo alto en muestra1 (ejemplo simple con grep -E) ---
BRCA1	120	98
TP53	340	310
MYC	670	590
GAPDH	5200	5100
ACTB	4980	5050

--- Buscar archivos .txt en todo el proyecto ---
../raw_data/conteos_simulados.txt
../raw_data/muestra1.txt
../results/muestra1_copia.txt


### 4.5 Redirección y tuberías (`pipes`)

Uno de los conceptos más potentes de la línea de comandos: encadenar comandos.

- `comando > archivo`: redirige la salida a un archivo (sobrescribe).
- `comando >> archivo`: agrega (append) al final del archivo.
- `comando1 | comando2`: la salida de `comando1` se pasa como entrada a `comando2`.

Comandos útiles para combinar en tuberías: `sort`, `uniq`, `cut`, `wc`, `grep`, `awk`, `sed`.


In [7]:
%%bash
cd ~/curso_bioinfo/clase1/proyecto_demo/raw_data

echo "--- Extraer solo la columna de nombres de gen (cut) ---"
cut -f1 conteos_simulados.txt

echo -e "\n--- Ordenar genes alfabéticamente y guardar en un nuevo archivo ---"
tail -n +2 conteos_simulados.txt | sort > genes_ordenados.txt
cat genes_ordenados.txt

echo -e "\n--- Combinando: contar cuántos genes hay en total ---"
tail -n +2 conteos_simulados.txt | wc -l


--- Extraer solo la columna de nombres de gen (cut) ---
gen_id
BRCA1
TP53
EGFR
MYC
GAPDH
ACTB

--- Ordenar genes alfabéticamente y guardar en un nuevo archivo ---
ACTB	4980	5050
BRCA1	120	98
EGFR	45	52
GAPDH	5200	5100
MYC	670	590
TP53	340	310

--- Combinando: contar cuántos genes hay en total ---
6


---
## 5. Permisos, procesos y variables de entorno

### 5.1 Permisos de archivos

Al listar con `ls -l` se observa algo como:

```
-rw-r--r--  1 usuario grupo  1240 ago  4 10:00 conteos_simulados.txt
```

Se interpreta en 4 bloques:

1. Tipo de archivo (`-` archivo normal, `d` directorio, `l` enlace simbólico).
2. Permisos del **dueño** (`rw-`).
3. Permisos del **grupo** (`r--`).
4. Permisos de **otros** (`r--`).

Donde `r` = leer, `w` = escribir, `x` = ejecutar.

Se modifican con `chmod`:

```bash
chmod u+x script.sh     # dar permiso de ejecución al dueño
chmod 755 script.sh     # notación numérica: rwxr-xr-x
```


In [8]:
%%bash
cd ~/curso_bioinfo/clase1/proyecto_demo/scripts

echo "#!/bin/bash" > analisis.sh
echo 'echo "Corriendo analisis..."' >> analisis.sh

echo "--- Permisos antes ---"
ls -l analisis.sh

chmod u+x analisis.sh

echo -e "\n--- Permisos después de chmod u+x ---"
ls -l analisis.sh

echo -e "\n--- Ejecutando el script ---"
./analisis.sh


--- Permisos antes ---
-rw-r--r-- 1 root root 41 Aug  3 22:01 analisis.sh

--- Permisos después de chmod u+x ---
-rwxr--r-- 1 root root 41 Aug  3 22:01 analisis.sh

--- Ejecutando el script ---
Corriendo analisis...


### 5.2 Variables de entorno

Las variables de entorno configuran el comportamiento del sistema y de los programas. Algunas importantes:

- `$HOME`: directorio home del usuario.
- `$PATH`: lista de directorios donde el sistema busca ejecutables.
- `$USER`: nombre de usuario actual.

Se consultan con `echo $NOMBRE` y se listan todas con `env` o `printenv`.


In [9]:
%%bash
echo "HOME: $HOME"
echo "USER: $USER"
echo "PATH: $PATH"


HOME: /root
USER: 
PATH: /opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin


### 5.3 Procesos básicos

- `ps` / `ps aux`: ver procesos en ejecución.
- `top` / `htop`: monitor interactivo de recursos (no siempre disponible dentro de un notebook, sí en su terminal SSH).
- `comando &`: correr un comando en segundo plano.
- `jobs`: ver trabajos en segundo plano de la sesión actual.
- `kill PID`: terminar un proceso por su identificador (PID).
- `nohup comando &`: correr un proceso que sobreviva al cierre de la sesión SSH (muy útil para análisis largos).


In [10]:
%%bash
# Listar procesos del sistema (columnas resumidas)
ps aux | head -n 10


USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0    984   560 ?        Ss   21:45   0:00 /sbin/docker-init -- /datalab/run.sh
root           7  0.0  0.4 1095248 57864 ?       Sl   21:45   0:00 /tools/node/bin/node /datalab/web/app.js
root          20  0.0  0.0   7372  3396 ?        S    21:45   0:00 /bin/bash -e /usr/local/colab/bin/oom_monitor.sh
root          22  0.0  0.0   7372  1924 ?        S    21:45   0:00 /bin/bash -e /datalab/run.sh
root          23  0.0  0.2 1286892 34108 ?       Sl   21:45   0:00 /usr/colab/bin/kernel_manager_proxy --listen_port=6000 --target_port=9000 --logtostderr --listen_host=172.28.0.12 --target_host=172.28.0.12 --tunnel_background_save_url=https://colab.research.google.com/tun/m/cc48301118ce562b961b3c22d803539adc1e0c19/m-s-kkb-use1b0-29oao4ntmdonx --tunnel_background_save_delay=10s --tunnel_periodic_background_save_frequency=30m0s --enable_output_coalescing=true --output_coalescing_required=true --us

---
## 6. Ejercicio integrador (25 min)

Trabajen en parejas. Usando únicamente la línea de comandos, completen los siguientes pasos dentro de `~/curso_bioinfo/clase1/`:

1. Creen una estructura de proyecto llamada `ejercicio` con subdirectorios `raw_data`, `scripts` y `results`.
2. Dentro de `raw_data`, generen un archivo `muestras.tsv` con al menos 8 filas (encabezado + 7 genes) similar al ejemplo de la sección 4.3, pero inventando sus propios valores.
3. Usando `grep`, encuentren todos los genes cuyo conteo en la muestra 1 sea mayor a 3 dígitos.
4. Usando `cut` y `sort`, generen un archivo `results/genes_ordenados.tsv` con los nombres de gen ordenados alfabéticamente.
5. Cuenten cuántas líneas tiene el archivo original con `wc -l`.
6. Creen un script `scripts/resumen.sh` que imprima "Resumen generado" y denle permisos de ejecución con `chmod`.
7. Ejecuten el script.

Usen la celda de abajo como borrador (o practiquen directamente en su terminal si ya tienen acceso SSH al clúster).


In [11]:
%%bash
cd ~/curso_bioinfo/clase1

# Escriban aquí sus comandos para el ejercicio integrador


---
## 7. Cierre y resumen

### Comandos vistos hoy

```
pwd, ls, cd, mkdir, touch, cp, mv, rm, rmdir
cat, less, head, tail, wc
grep, find, cut, sort, uniq
chmod, ps, echo, env
ssh, ssh-keygen, ssh-copy-id
```

### Cheatsheet rápido

| Necesito... | Comando |
|---|---|
| Saber dónde estoy | `pwd` |
| Ver archivos | `ls -la` |
| Moverme a un directorio | `cd ruta` |
| Crear directorio | `mkdir -p ruta` |
| Ver contenido de un archivo | `cat`, `less`, `head`, `tail` |
| Buscar texto dentro de archivos | `grep patrón archivo` |
| Buscar archivos por nombre | `find . -name "*.ext"` |
| Combinar comandos | `comando1 \| comando2` |
| Guardar salida en archivo | `comando > archivo` |
| Ver/cambiar permisos | `ls -l`, `chmod` |
| Conectarse a un servidor | `ssh usuario@servidor` |

### Para la próxima clase (Clase 2 — Ambiente Linux: Bash intermedio)

- Confirmar que todos pudieron conectarse al clúster de la UDD vía SSH.
- Repasar los comandos de esta clase; practicar en su propia terminal, no solo en el notebook.
- Leer sobre: scripts de bash, variables, bucles `for`, y control de flujo (`if`) — se profundizará en la próxima sesión.

### Recursos adicionales

- `man <comando>` — manual de cualquier comando (ej. `man grep`).
- `<comando> --help` — ayuda rápida.
- [The Linux Command Line (libro gratuito, W. Shotts)](https://linuxcommand.org/tlcl.php)
- [Software Carpentry — The Unix Shell](https://swcarpentry.github.io/shell-novice/)
